In [ ]:
import pandas as pd
import numpy as np
import ast

In [3]:
# --- 1. THE ATTRIBUTE PARSER FUNCTION ---
def parse_yelp_attributes(attr_dict):
    if not isinstance(attr_dict, dict):
        return ""
    
    extracted_features = []
    
    if 'NoiseLevel' in attr_dict:
        noise = attr_dict['NoiseLevel'].replace("u'", "").replace("'", "")
        if noise != 'None':
            extracted_features.append(f"Noise level is {noise}.")
            
    if 'Alcohol' in attr_dict:
        alc = attr_dict['Alcohol'].replace("u'", "").replace("'", "")
        if alc == 'full_bar' or alc == 'beer_and_wine':
            extracted_features.append(f"Serves {alc.replace('_', ' ')}.")
            
    if 'Ambience' in attr_dict:
        try:
            ambience_dict = ast.literal_eval(attr_dict['Ambience'])
            if isinstance(ambience_dict, dict):
                true_vibes = [vibe for vibe, is_true in ambience_dict.items() if is_true]
                if true_vibes:
                    extracted_features.append(f"Ambience is {', '.join(true_vibes)}.")
        except:
            pass 
            
    return " ".join(extracted_features)

In [4]:
# --- 2. PROCESS BUSINESS DATA & APPLY CITY SWAP ---
print("Loading and augmenting business data...")
biz_df = pd.read_json('yelp/yelp_academic_dataset_business.json', lines=True)

# Filter for Philly and Restaurants/Bars
philly_biz = biz_df[biz_df['city'] == 'Philadelphia'].copy()
philly_biz = philly_biz.dropna(subset=['categories'])
philly_biz = philly_biz[philly_biz['categories'].str.contains('Restaurants|Food|Bars')]

# Apply the text augmentation parser!
philly_biz['metadata_sentence'] = philly_biz['attributes'].apply(parse_yelp_attributes)

# THE CITY SWAP: Generate random valid NYC Coordinates
np.random.seed(42)
philly_biz['lat'] = np.random.uniform(40.50, 40.90, len(philly_biz))
philly_biz['lon'] = np.random.uniform(-74.25, -73.70, len(philly_biz))

# Keep only what we need
philly_biz = philly_biz[['business_id', 'name', 'lat', 'lon', 'metadata_sentence']]
valid_business_ids = set(philly_biz['business_id'])

print(f"Augmented {len(philly_biz)} venues and mapped them to NYC.")

Loading and augmenting business data...
Augmented 7353 venues and mapped them to NYC.


In [5]:
# --- 3. PROCESS 5GB REVIEW FILE IN CHUNKS ---
print("Processing 5GB review file in chunks... (This takes a few minutes)")
chunk_size = 100000
filtered_reviews_list = []
chunks_processed = 0

for chunk in pd.read_json('/Users/alinazim/Downloads/yelp_dataset/yelp_academic_dataset_review.json', lines=True, chunksize=chunk_size):
    # Keep only reviews for our Philly businesses
    matched_reviews = chunk[chunk['business_id'].isin(valid_business_ids)]
    
    # Keep only text and stars
    clean_chunk = matched_reviews[['business_id', 'stars', 'text']]
    filtered_reviews_list.append(clean_chunk)
    
    chunks_processed += 1
    if chunks_processed % 10 == 0:
        print(f"Processed {chunks_processed * chunk_size:,} rows...")

all_filtered_reviews = pd.concat(filtered_reviews_list, ignore_index=True)
print(f"Extraction complete! Kept {len(all_filtered_reviews):,} reviews.")

Processing 5GB review file in chunks... (This takes a few minutes)
Processed 1,000,000 rows...
Processed 2,000,000 rows...
Processed 3,000,000 rows...
Processed 4,000,000 rows...
Processed 5,000,000 rows...
Processed 6,000,000 rows...
Processed 7,000,000 rows...
Extraction complete! Kept 755,335 reviews.


In [6]:
# --- 4. MERGE AND CREATE THE FINAL AUGMENTED TEXT ---
print("Fusing metadata with review text...")
final_nlp_dataset = pd.merge(all_filtered_reviews, philly_biz, on='business_id', how='inner')

# THE MAGIC LINE: Glue the metadata sentence to the front of every single review
final_nlp_dataset['augmented_text'] = final_nlp_dataset['metadata_sentence'] + " " + final_nlp_dataset['text']

# Clean up the dataframe to save memory
final_nlp_dataset = final_nlp_dataset[['business_id', 'name', 'lat', 'lon', 'stars', 'augmented_text']]

# Save the final ML-ready file
final_nlp_dataset.to_csv('sweetspot_nlp_training_data.csv', index=False)
print("\nSuccess! Saved 'sweetspot_nlp_training_data.csv'.")
display(final_nlp_dataset.head(3))

Fusing metadata with review text...

Success! Saved 'sweetspot_nlp_training_data.csv'.


,business_id,name,lat,lon,stars,augmented_text
0,kxX2SOes4o-D3ZQBkiMRfA,Zaika,40.600097,-73.764440,5,"Noise level is average. Wow! Yummy, different..."
1,04UD14gamNjLY0IDYVhHJg,Dmitri's,40.670843,-73.770646,1,Noise level is average. I am a long term frequ...
2,RZtGWDLCAtuipwaZ-UfjmQ,LaScala's,40.804604,-73.730387,4,Noise level is average. Serves full bar. Good ...


In [7]:
display(final_nlp_dataset.head(50))

,business_id,name,lat,lon,stars,augmented_text
0,kxX2SOes4o-D3ZQBkiMRfA,Zaika,40.600097,-73.764440,5,"Noise level is average. Wow! Yummy, different..."
1,04UD14gamNjLY0IDYVhHJg,Dmitri's,40.670843,-73.770646,1,Noise level is average. I am a long term frequ...
2,RZtGWDLCAtuipwaZ-UfjmQ,LaScala's,40.804604,-73.730387,4,Noise level is average. Serves full bar. Good ...
3,YtSqYv1Q_pOltsVPSx54SA,Rittenhouse Grill,40.603298,-73.921802,5,Noise level is average. Serves full bar. Treme...
4,eFvzHawVJofxSnD7TgbZtg,Good Karma Cafe,40.671274,-74.095409,5,Noise level is average. My absolute favorite c...
5,kq5Ghhh14r-eCxlVmlyd8w,The Coventry Deli,40.624684,-74.112554,5,Noise level is average. My boyfriend and I tri...
6,oBhJuukGRqPVvYBfTkhuZA,Square 1682,40.544021,-73.882921,4,Noise level is average. Serves full bar. The o...
7,EtKSTHV5Qx_Q7Aur9o4kQQ,Village Whiskey,40.708123,-74.128369,5,Noise level is average. Serves full bar. On a ...
8,VJEzpfLs_Jnzgqh5A_FVTg,Jasmine Rice - Rittenhouse,40.512200,-74.105519,4,Noise level is average. It was my fiance's bir...
9,NQSnr4RPUScss607oxOaqw,Chase's Hop Shop,40.535397,-73.921586,5,Noise level is quiet. Serves beer and wine. My...
